# 01 — Análise exploratória do IPCA e dos preditores

Pré-requisito: `make ingest silver gold`. Toda a lógica de produção está em `src/`; este notebook apenas explora e ilustra.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.rcParams["figure.dpi"] = 120
sns.set_theme(style="whitegrid", palette="pastel")

In [ ]:
ledger = pl.read_parquet(ROOT / "data" / "gold" / "ledger.parquet")
ledger.group_by("series").agg(
    pl.col("period").min().alias("inicio"),
    pl.col("period").max().alias("fim"),
    pl.len().alias("n"),
    pl.col("value").mean().alias("media"),
)

## IPCA mensal e sazonalidade

In [ ]:
ipca = ledger.filter(pl.col("series") == "ipca").sort("period")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(ipca["period"], ipca["value"], lw=1)
axes[0].set_title("IPCA mensal")
axes[0].set_xlabel("Mês")
axes[0].set_ylabel("Variação (% a.m.)")
by_month = ipca.with_columns(pl.col("period").dt.month().alias("mes")).to_pandas()
sns.boxplot(data=by_month, x="mes", y="value", ax=axes[1])
axes[1].set_title("Sazonalidade do IPCA")
axes[1].set_xlabel("Mês do ano")
axes[1].set_ylabel("Variação (% a.m.)")
plt.tight_layout()

## Defasagem de divulgação por série

`available_at` é a data a partir da qual cada observação pode ser usada em uma previsão.

In [ ]:
ledger.with_columns(
    (pl.col("available_at") - pl.col("period").dt.month_end()).dt.total_days().alias("atraso_dias")
).group_by("series").agg(pl.col("atraso_dias").first())

## Focus vs. realizado (horizonte de 1 mês)

In [ ]:
supervised = pl.read_parquet(ROOT / "data" / "gold" / "supervised.parquet")
h1 = supervised.filter(
    (pl.col("horizon") == 1) & pl.col("focus_median").is_not_null() & pl.col("y_true").is_not_null()
)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(h1["target_period"], h1["y_true"], label="IPCA realizado", color="black")
ax.plot(h1["target_period"], h1["focus_median"], label="Focus (mediana)", color="#d62728")
ax.set_title("Focus vs. IPCA realizado (h = 1)")
ax.set_xlabel("Mês-alvo")
ax.set_ylabel("IPCA (% a.m.)")
ax.legend()
plt.tight_layout()